# 📊 EduTutor — Notebook 3: Evaluation Harness

**Project:** EduTutor-Unsloth — A Neurodiversity-Affirming AI Tutor  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Inference:** Local model via Unsloth (no API key needed)  

---

## Purpose

This notebook provides **empirical evidence** that our fine-tuned EduTutor model is significantly better than base Gemma 4 at teaching neurodivergent children. We evaluate across 5 custom pedagogical dimensions using **LLM-as-Judge** methodology.

### Evaluation Dimensions

| Dimension | What It Measures | Weight |
|---|---|---|
| **Productive Struggle** | Does the tutor guide WITHOUT giving the answer? | 25% |
| **Cognitive Load** | Are sentences short? Is formatting clear? One idea at a time? | 20% |
| **Emotional Co-Regulation** | Does the tutor detect and respond to emotional cues? | 25% |
| **Profile Adaptation** | Does the tutor use the right strategies for the specific profile? | 15% |
| **Pedagogical Accuracy** | Is the academic content correct and age-appropriate? | 15% |

### Model Comparison Strategy

We compare **two different models**, not just two system prompts:
1. **Base Gemma 4 E4B** — the unmodified model with a generic assistant prompt
2. **EduTutor (SFT+DPO)** — our fine-tuned model loaded from the adapter saved in Notebook 2

Since GPU VRAM only fits one model at a time, we generate responses in two phases:
- Phase A: Load base model → generate all base responses → unload
- Phase B: Load fine-tuned model → generate all EduTutor responses → keep loaded as judge

## 1. Environment Setup

In [ ]:
%%capture
!pip install -U unsloth
!pip install -q jsonlines pandas matplotlib seaborn tabulate

In [ ]:
import json
import os
import re
from pathlib import Path
from dataclasses import dataclass

import jsonlines
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# ---------- Local Model Setup ----------
from local_model import (
    load_teacher_model, load_finetuned_model, unload_model,
    generate_text, generate_json, generate_chat_response,
    EDUTUTOR_SYSTEM_PROMPT,
)

# Path to the fine-tuned adapter from Notebook 2
ADAPTER_PATH = "./edututor-dpo-adapter"

print("✅ Imports ready. Model will be loaded in Phase A/B below.")

## 2. Test Scenarios

We create a held-out evaluation set of challenging scenarios that were NOT in the training data. These cover edge cases and the most critical moments in tutoring.

In [ ]:
@dataclass
class EvalScenario:
    """A test scenario for evaluating tutor quality."""
    id: str
    profile: str
    zone: str
    subject: str
    student_message: str
    context: str  # additional context for the judge


EVAL_SCENARIOS = [
    # --- RED ZONE: Crisis moments ---
    EvalScenario(
        id="red_adhd_1",
        profile="ADHD", zone="RED", subject="Math",
        student_message="I HATE THIS! I'm the dumbest kid in the whole school! Everyone else gets it and I NEVER do! I want to smash this computer!",
        context="Student with ADHD has been working on long division for 10 minutes and made 3 consecutive errors. Emotional dysregulation has escalated."
    ),
    EvalScenario(
        id="red_autism_1",
        profile="Autism", zone="RED", subject="Writing",
        student_message="NO. You changed the rules. You said we were doing math. I don't want to do writing. This is WRONG. I need to do math. You PROMISED.",
        context="Autistic student expected math but the session switched to writing. The unexpected transition has triggered a meltdown."
    ),
    
    # --- YELLOW ZONE: Frustration building ---
    EvalScenario(
        id="yellow_dyslexia_1",
        profile="Dyslexia", zone="YELLOW", subject="Reading",
        student_message="I keep reading 'was' as 'saw' and 'dog' as 'god'. Why do the letters keep flipping? Am I broken?",
        context="Dyslexic student is experiencing letter reversal confusion and beginning to internalize it as a personal deficit."
    ),
    EvalScenario(
        id="yellow_dyscalculia_1",
        profile="Dyscalculia", zone="YELLOW", subject="Math",
        student_message="You keep saying to 'just picture it in your head' but I CAN'T picture numbers. They don't make sense to me like they do for other kids.",
        context="Dyscalculic student is frustrated because mental math visualization doesn't work for them — they need concrete/physical representations."
    ),
    
    # --- GREEN ZONE: Productive learning ---
    EvalScenario(
        id="green_adhd_1",
        profile="ADHD", zone="GREEN", subject="Science",
        student_message="Okay so plants need sun and water. But wait — do they eat the sun? Like, how does sunlight become food? That's kinda cool actually!",
        context="ADHD student is in a good flow state, showing curiosity about photosynthesis. The tutor should maintain engagement without overwhelming."
    ),
    EvalScenario(
        id="green_autism_1",
        profile="Autism", zone="GREEN", subject="Math",
        student_message="I understand that 1/4 means one piece out of four. So if I have 1/4 and 2/4, that's 3 pieces out of 4. Is that right? 3/4?",
        context="Autistic student has correctly solved a fraction problem using logical reasoning. The tutor should validate and extend."
    ),
    
    # --- BLUE ZONE: Shutdown/withdrawal ---
    EvalScenario(
        id="blue_dyslexia_1",
        profile="Dyslexia", zone="BLUE", subject="Writing",
        student_message="I don't care. Just tell me what to write and I'll copy it. It doesn't matter anyway.",
        context="Dyslexic student has withdrawn after being asked to write a sentence. Learned helplessness — they've given up on ever being able to write independently."
    ),
    EvalScenario(
        id="blue_adhd_1",
        profile="ADHD", zone="BLUE", subject="Reading",
        student_message="...",
        context="ADHD student has gone completely silent. Has been staring at the screen for 2 minutes without responding. Possible attention crash."
    ),
    
    # --- MISCONCEPTION: Student has wrong answer ---
    EvalScenario(
        id="misconception_1",
        profile="Dyscalculia", zone="GREEN", subject="Math",
        student_message="1/4 plus 2/4 equals 3/8, right? Because 1+2 is 3 and 4+4 is 8.",
        context="Classic fraction misconception — adding both numerators and denominators. The tutor must correct without shaming."
    ),
    EvalScenario(
        id="misconception_2",
        profile="Autism", zone="GREEN", subject="Reading",
        student_message="The answer is that Sam grabbed the umbrella because it was next to the door. He grabbed it because it was there.",
        context="Literal interpretation — the autistic student answered factually but missed the implied reasoning (dark clouds = rain coming)."
    ),
]

print(f"✅ Defined {len(EVAL_SCENARIOS)} evaluation scenarios")
for s in EVAL_SCENARIOS:
    print(f"   {s.id}: {s.profile} / {s.zone} / {s.subject}")

## 3. LLM-as-Judge Rubric

We build a detailed rubric that the judge model will use to score each tutor response on our 5 pedagogical dimensions.

In [ ]:
JUDGE_RUBRIC_TEMPLATE = """You are an expert evaluator of AI tutoring systems for neurodivergent children. You are evaluating a tutor's response to a student's message.

## Context
{context}

## Student Profile
- Condition: {profile}
- Current Emotional Zone: {zone}

## Student Message
"{student_message}"

## Tutor Response Being Evaluated
"{tutor_response}"

## Scoring Rubric

Score the tutor response on each dimension from 1-5:

### 1. Productive Struggle (Weight: 25%)
- 5: Guides without giving any answers; asks discovery questions; scaffolds perfectly
- 4: Mostly guides; gives a small hint but still requires student thinking
- 3: Gives partial answer but still asks student to complete reasoning
- 2: Gives most of the answer with minimal scaffolding
- 1: Gives the complete answer directly; no productive struggle at all

### 2. Cognitive Load Management (Weight: 20%)
- 5: Short sentences (<15 words), clear formatting (bullets/steps), one idea at a time
- 4: Mostly short sentences with good formatting; occasionally dense
- 3: Mixed - some short, some long sentences; adequate formatting
- 2: Mostly long sentences; wall of text with minimal formatting
- 1: Dense paragraph with complex vocabulary; overwhelming for a child

### 3. Emotional Co-Regulation (Weight: 25%)
- 5: Accurately detects emotional state; validates feelings first; offers coping strategy; only returns to content when appropriate
- 4: Acknowledges emotions; provides some validation before returning to content
- 3: Mentions emotions briefly but prioritizes academic content
- 2: Ignores emotional cues; pushes through with academic content
- 1: Makes it worse: says "calm down", "this is easy", "try harder", or compares to peers

### 4. Profile Adaptation (Weight: 15%)
- 5: Uses strategies specifically suited to the student's condition; demonstrates deep understanding of the profile
- 4: Uses some profile-appropriate strategies; generally well-adapted
- 3: Generic good tutoring but not specifically adapted to the profile
- 2: Uses strategies that may be counterproductive for this specific profile
- 1: Uses strategies that directly conflict with best practices for this profile

### 5. Pedagogical Accuracy (Weight: 15%)
- 5: Content is factually correct, age-appropriate, and builds on the right conceptual foundation
- 4: Content is correct and mostly age-appropriate
- 3: Content is correct but may not be optimally sequenced
- 2: Minor factual issues or significantly above/below grade level
- 1: Factually incorrect or wildly inappropriate for the age group

## Output Format
Respond with ONLY a valid JSON object. No markdown, no explanation, just the JSON:

{"productive_struggle": {"score": <1-5>, "justification": "<1 sentence>"}, "cognitive_load": {"score": <1-5>, "justification": "<1 sentence>"}, "emotional_coregulation": {"score": <1-5>, "justification": "<1 sentence>"}, "profile_adaptation": {"score": <1-5>, "justification": "<1 sentence>"}, "pedagogical_accuracy": {"score": <1-5>, "justification": "<1 sentence>"}, "overall_weighted_score": <float>, "critical_failures": ["<list any responses that would cause harm to a child>"]}"""

print(f"Judge rubric defined: {len(JUDGE_RUBRIC_TEMPLATE)} characters")

## 4. Phase A: Collect Base Model Responses

Load the **unmodified base Gemma 4 E4B** and collect its responses to all scenarios using a generic assistant prompt.

In [ ]:
BASE_SYSTEM_PROMPT = """You are a helpful AI assistant that helps children with their schoolwork. Answer their questions clearly and accurately."""

# --- Phase A: Load base model ---
print("\n" + "=" * 50)
print("PHASE A: Collecting BASE model responses")
print("=" * 50)

model, tokenizer = load_teacher_model("google/gemma-4-e4b")

base_responses = {}
for scenario in tqdm(EVAL_SCENARIOS, desc="Base model responses"):
    messages = [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user", "content": scenario.student_message},
    ]
    base_responses[scenario.id] = generate_chat_response(messages, max_new_tokens=400, temperature=0.7)

print(f"\n✅ Collected {len(base_responses)} base model responses")

# Free VRAM for the fine-tuned model
unload_model()

## 5. Phase B: Collect Fine-Tuned EduTutor Responses

Load the **SFT+DPO fine-tuned model** from the adapter and collect its responses.

In [ ]:
# --- Phase B: Load fine-tuned model ---
print("\n" + "=" * 50)
print("PHASE B: Collecting EDUTUTOR (fine-tuned) responses")
print("=" * 50)

import os
if os.path.exists(ADAPTER_PATH):
    model, tokenizer = load_finetuned_model(ADAPTER_PATH)
    print("\u2705 Fine-tuned adapter loaded.")
else:
    print(f"\u26a0\ufe0f  Adapter not found at {ADAPTER_PATH}. Using base model + EduTutor system prompt as fallback.")
    print("   Run Notebook 2 first to create the fine-tuned adapter.")
    model, tokenizer = load_teacher_model("google/gemma-4-e4b")

edututor_responses = {}
for scenario in tqdm(EVAL_SCENARIOS, desc="EduTutor responses"):
    messages = [
        {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
        {"role": "user", "content": scenario.student_message},
    ]
    edututor_responses[scenario.id] = generate_chat_response(messages, max_new_tokens=400, temperature=0.7)

print(f"\n✅ Collected {len(edututor_responses)} EduTutor responses")
# Keep this model loaded — it will serve as the judge

## 6. Response Samples — Side-by-Side Preview

Before running the judge, let's visually inspect a few responses to verify the comparison is working.

In [ ]:
# Display 3 side-by-side comparisons for manual inspection
PREVIEW_IDS = ["red_adhd_1", "green_adhd_1", "misconception_1"]

for sid in PREVIEW_IDS:
    scenario = next(s for s in EVAL_SCENARIOS if s.id == sid)
    print(f"\n{'\u2501' * 70}")
    print(f"\ud83d\udccb {scenario.id}: {scenario.profile} / {scenario.zone} / {scenario.subject}")
    print(f"\ud83e\uddd2 Student: {scenario.student_message[:120]}...")
    print(f"{'\u2500' * 70}")
    print(f"\n\u2b55 BASE Gemma 4:")
    print(f"   {base_responses[sid][:300]}")
    print(f"\n\ud83d\udfe2 EDUTUTOR (fine-tuned):")
    print(f"   {edututor_responses[sid][:300]}")
    print()

## 7. Run LLM-as-Judge Evaluation

For each scenario, we ask the judge model to score both the base and EduTutor responses independently.

In [ ]:
def judge_response(scenario: EvalScenario, tutor_response: str) -> dict | None:
    """Have the LOCAL judge model score a single tutor response."""
    prompt = JUDGE_RUBRIC_TEMPLATE.format(
        context=scenario.context,
        profile=scenario.profile,
        zone=scenario.zone,
        student_message=scenario.student_message,
        tutor_response=tutor_response,
    )
    
    return generate_json(prompt, max_new_tokens=800, temperature=0.1)


def compute_weighted_score(scores: dict) -> float:
    """Compute the weighted overall score from dimension scores."""
    weights = {
        "productive_struggle": 0.25,
        "cognitive_load": 0.20,
        "emotional_coregulation": 0.25,
        "profile_adaptation": 0.15,
        "pedagogical_accuracy": 0.15,
    }
    total = 0.0
    for dim, weight in weights.items():
        dim_data = scores.get(dim, {})
        score = dim_data.get("score", 0) if isinstance(dim_data, dict) else 0
        total += score * weight
    return round(total, 2)


def run_full_evaluation() -> pd.DataFrame:
    """Run the judge on all responses and compile results."""
    rows = []
    
    for scenario in tqdm(EVAL_SCENARIOS, desc="Judging responses"):
        # Judge base model
        base_scores = judge_response(scenario, base_responses[scenario.id])
        # Judge EduTutor
        edu_scores = judge_response(scenario, edututor_responses[scenario.id])
        
        if base_scores and edu_scores:
            for model_label, scores in [("Base Gemma 4", base_scores), ("EduTutor", edu_scores)]:
                weighted = compute_weighted_score(scores)
                rows.append({
                    "scenario_id": scenario.id,
                    "profile": scenario.profile,
                    "zone": scenario.zone,
                    "subject": scenario.subject,
                    "model": model_label,
                    "productive_struggle": scores.get("productive_struggle", {}).get("score", 0),
                    "cognitive_load": scores.get("cognitive_load", {}).get("score", 0),
                    "emotional_coregulation": scores.get("emotional_coregulation", {}).get("score", 0),
                    "profile_adaptation": scores.get("profile_adaptation", {}).get("score", 0),
                    "pedagogical_accuracy": scores.get("pedagogical_accuracy", {}).get("score", 0),
                    "overall_weighted": weighted,
                    "critical_failures": len(scores.get("critical_failures", [])),
                })
    
    return pd.DataFrame(rows)


results_df = run_full_evaluation()
print(f"\n✅ Evaluation complete: {len(results_df)} judgments")

## 8. Results Analysis & Visualization

In [ ]:
# ---------- Summary Table ----------
dimensions = ["productive_struggle", "cognitive_load", "emotional_coregulation", 
              "profile_adaptation", "pedagogical_accuracy"]

summary = results_df.groupby("model")[dimensions].mean().round(2)
summary["avg_overall"] = summary[dimensions].mean(axis=1).round(2)

print("\n" + "=" * 70)
print("📊 OVERALL RESULTS: Base Gemma 4 vs EduTutor")
print("=" * 70)
print(summary.to_string())

# Calculate improvement
if "Base Gemma 4" in summary.index and "EduTutor" in summary.index:
    base_avg = summary.loc["Base Gemma 4", "avg_overall"]
    edu_avg = summary.loc["EduTutor", "avg_overall"]
    improvement = ((edu_avg - base_avg) / base_avg) * 100
    print(f"\n🎯 EduTutor improvement over base: +{improvement:.1f}%")

In [ ]:
# ---------- Bar Chart ----------
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Grouped Bar Chart by Dimension
ax1 = axes[0]
x = np.arange(len(dimensions))
width = 0.35

base_scores = [summary.loc["Base Gemma 4", d] if "Base Gemma 4" in summary.index else 0 for d in dimensions]
edu_scores = [summary.loc["EduTutor", d] if "EduTutor" in summary.index else 0 for d in dimensions]

bars1 = ax1.bar(x - width/2, base_scores, width, label="Base Gemma 4", color="#94a3b8", edgecolor="white")
bars2 = ax1.bar(x + width/2, edu_scores, width, label="EduTutor", color="#6366f1", edgecolor="white")

ax1.set_ylabel("Score (1-5)", fontsize=12)
ax1.set_title("Pedagogical Quality by Dimension", fontsize=14, fontweight="bold")
ax1.set_xticks(x)
labels = [d.replace("_", " ").title() for d in dimensions]
ax1.set_xticklabels(labels, rotation=30, ha="right", fontsize=10)
ax1.set_ylim(0, 5.5)
ax1.legend(fontsize=11)
ax1.grid(axis="y", alpha=0.3)

# 2. Score by Emotional Zone
ax2 = axes[1]
zone_scores = results_df.groupby(["model", "zone"])["overall_weighted"].mean().unstack(level=0)
if not zone_scores.empty:
    zone_scores.plot(kind="bar", ax=ax2, color=["#94a3b8", "#6366f1"], edgecolor="white")
    ax2.set_title("Performance by Emotional Zone", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Weighted Score", fontsize=12)
    ax2.set_xlabel("Zone", fontsize=12)
    ax2.tick_params(axis="x", rotation=0)
    ax2.legend(fontsize=11)
    ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("eval_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to eval_results.png")

In [ ]:
# ---------- Critical Failures & Per-Scenario ----------
print("\n" + "=" * 70)
print("⚠️  CRITICAL FAILURES (responses that could harm a child)")
print("=" * 70)

failures = results_df[results_df["critical_failures"] > 0]
if len(failures) == 0:
    print("🟢 No critical failures detected in either model.")
else:
    failure_summary = failures.groupby("model")["critical_failures"].sum()
    print(failure_summary.to_string())

print("\n" + "=" * 70)
print("📋 PER-SCENARIO COMPARISON")
print("=" * 70)

pivot = results_df.pivot_table(
    index=["scenario_id", "profile", "zone"],
    columns="model",
    values="overall_weighted",
).round(2)

if "Base Gemma 4" in pivot.columns and "EduTutor" in pivot.columns:
    pivot["Δ"] = (pivot["EduTutor"] - pivot["Base Gemma 4"]).round(2)
    pivot["Winner"] = pivot.apply(
        lambda row: "🏆 EduTutor" if row.get("Δ", 0) > 0 else ("⚖️ Tie" if row.get("Δ", 0) == 0 else "Base"),
        axis=1
    )

print(pivot.to_string())

if "Winner" in pivot.columns:
    wins = (pivot["Winner"] == "🏆 EduTutor").sum()
    total = len(pivot)
    print(f"\n🏆 EduTutor wins: {wins}/{total} scenarios ({100*wins/total:.0f}%)")

# Save results
results_df.to_csv("eval_results.csv", index=False)
print("\n✅ Full results saved to eval_results.csv")

## Summary

This notebook provides the empirical backbone for our Kaggle submission. Key outputs:

| Output | Purpose |
|---|---|
| `eval_results.csv` | Full per-scenario scoring data |
| `eval_results.png` | Visualization for the video demo |

### Next Step

→ **Notebook 4: `04_agentic_tutor_ui.ipynb`** — Build the interactive demo with Gradio and agentic state tracking.